In [6]:
import os
import pickle
import numpy as np
from create_data import create_dataset
from create_train_test_splits import Splits

# Create training and analysis datasets

This worksheet outlines the steps required to generate the data needed for model training and subsequent analysis. The input is the experiment h5ad file, containing the gene expression data, the cell metadata, and gene data. It transforms them into three files needed for model training: 1) a numpy memmap file (.dat file) containing the gene expression data, a 2) .pkl file containing the cell metadata, and a 3) .pkl containing the train/test splits. The numpy memmap format allows for rapid fiel reading, greatly speeding up model training.

## Create PD Myeloid data

In [2]:
base_dir = "/sc/arion/projects/CommonMind/amppd/snRNAseq/taxonomy/freeze2_0/"
source_paths = [
    base_dir + "AMPPD_freeze2_7_cx_rareMut_2024_12_16.h5ad",
]
target_path = "/sc/arion/projects/psychAD/massen06/PD_myeloid"

if not os.path.isdir(target_path):
    os.mkdir(target_path)

obs_keys = ["path_braak_lb", "Brain_bank", "Source", "age", "sex", "participant_id", 
            "brain_reg_w_gt", "n_counts", "n_genes", "RIN", "derived_class2_Dec2024",
           "derived_subclass2_Dec2024", "derived_subtype2_Dec2024"]

      
cell_restrictions = {"derived_class2_Dec2024": "Myeloid"}


In [3]:
create_dataset(source_paths, target_path, obs_keys,  cell_restrictions = cell_restrictions)

/sc/arion/projects/psychAD/massen06/backup_0430/vae/create_data.py:92: UserWarning: evaluating in Python space because the '*' operator is not supported by numexpr for the bool dtype, use '&' instead.
  cond *= self.anndata.obs[k] == v


Number of eligible cells: 118492
Number of genes selected: 17285
Size of anndata 2092520
Saving the metadata...
Saving the expression data in the memmap array...
Creating data. Number of cell: 118492, number of genes: 17285
Chunk number 0 out of 12 created
Chunk number 1 out of 12 created
Chunk number 2 out of 12 created
Chunk number 3 out of 12 created
Chunk number 4 out of 12 created
Chunk number 5 out of 12 created
Chunk number 6 out of 12 created
Chunk number 7 out of 12 created
Chunk number 8 out of 12 created
Chunk number 9 out of 12 created
Chunk number 10 out of 12 created
Chunk number 11 out of 12 created


### Add condensed Braak scores

While training the model on the original Braak stages (values 0 to 6) works reasonably well, we found that grouping the Braak stages into different groups (defined below), and training the model to classify the group, provided slightly greater accuracy.

In [7]:
# for path_braak_lb_condensed
# 0 -> 0, 1,2 -> 1, 3,4 -> 2, 5,6 -> 3

# for path_braak_lb_condensed_v2
# 0 -> 0, 1,2 -> 1, 3,4,5,6 -> 2

meta_fn = f"{target_path}/metadata.pkl"
meta = pickle.load(open(meta_fn, "rb"))

meta["obs"]["path_braak_lb_condensed"] = np.zeros_like(meta["obs"]["path_braak_lb"])
meta["obs"]["path_braak_lb_condensed_v2"] = np.zeros_like(meta["obs"]["path_braak_lb"])

idx = np.where((meta["obs"]["path_braak_lb"] >= 1) * (meta["obs"]["path_braak_lb"] <= 2))[0]
meta["obs"]["path_braak_lb_condensed"][idx] = 1
meta["obs"]["path_braak_lb_condensed_v2"][idx] = 1

idx = np.where((meta["obs"]["path_braak_lb"] >= 3) * (meta["obs"]["path_braak_lb"] <= 4))[0]
meta["obs"]["path_braak_lb_condensed"][idx] = 2
meta["obs"]["path_braak_lb_condensed_v2"][idx] = 2

idx = np.where((meta["obs"]["path_braak_lb"] >= 5) * (meta["obs"]["path_braak_lb"] <= 6))[0]
meta["obs"]["path_braak_lb_condensed"][idx] = 3
meta["obs"]["path_braak_lb_condensed_v2"][idx] = 2

pickle.dump(meta, open(meta_fn, "wb"))

In [3]:
meta_fn = f"{target_path}/metadata.pkl"
save_fn = f"{target_path}/train_test_20splits.pkl"

splits = Splits(meta_fn, save_fn, n_splits=20, seed=42)
splits.create_splits()

Split number 0
Size of train subjects/indices, test subjects/indices
Number of train subjects: 91, Number of test subjects: 5, Number of train indices: 109664, Number of test indices: 5600
Intersection size between train and test subjects: 0
Intersection size between train and test indices: 0
Split number 1
Size of train subjects/indices, test subjects/indices
Number of train subjects: 91, Number of test subjects: 5, Number of train indices: 112242, Number of test indices: 3022
Intersection size between train and test subjects: 0
Intersection size between train and test indices: 0
Split number 2
Size of train subjects/indices, test subjects/indices
Number of train subjects: 91, Number of test subjects: 5, Number of train indices: 112009, Number of test indices: 3255
Intersection size between train and test subjects: 0
Intersection size between train and test indices: 0
Split number 3
Size of train subjects/indices, test subjects/indices
Number of train subjects: 91, Number of test subj

## Create AD Immune data

In [4]:
base_dir = "/sc/arion/projects/psychAD/NPS-AD/freeze2_rc/h5ad_final/"
source_paths = [
    base_dir + "RUSH_2024-02-01_14_53.h5ad",
    base_dir + "MSSM_2024-02-01_16_17.h5ad",
]
target_path = "/sc/arion/projects/psychAD/massen06/AD_immune"

if not os.path.isdir(target_path):
    os.mkdir(target_path)

obs_keys = ['CERAD', 'BRAAK_AD', 'BRAAK_PD', 'Dementia', 'class', 'subclass', 'subtype',
            'Sex', 'Age', 'SubID',  'Brain_bank', "n_counts", "n_genes"]
cell_restrictions = {"class": "Immune"}


In [3]:
create_dataset(source_paths, target_path, obs_keys,  cell_restrictions = cell_restrictions)

gene_name matched between the first two datasets
percent_cells matched between the first two datasets


/sc/arion/projects/psychAD/massen06/backup_0430/vae/create_data.py:90: UserWarning: evaluating in Python space because the '*' operator is not supported by numexpr for the bool dtype, use '&' instead.
  cond *= self.anndata.obs[k] == v


Number of eligible cells: 258951
Number of genes selected: 17265
Size of anndata 4834135
Saving the metadata...
Saving the expression data in the memmap array...
Creating data. Number of cell: 258951, number of genes: 17265
Chunk number 0 out of 26 created
Chunk number 1 out of 26 created
Chunk number 2 out of 26 created
Chunk number 3 out of 26 created
Chunk number 4 out of 26 created
Chunk number 5 out of 26 created
Chunk number 6 out of 26 created
Chunk number 7 out of 26 created
Chunk number 8 out of 26 created
Chunk number 9 out of 26 created
Chunk number 10 out of 26 created
Chunk number 11 out of 26 created
Chunk number 12 out of 26 created
Chunk number 13 out of 26 created
Chunk number 14 out of 26 created
Chunk number 15 out of 26 created
Chunk number 16 out of 26 created
Chunk number 17 out of 26 created
Chunk number 18 out of 26 created
Chunk number 19 out of 26 created
Chunk number 20 out of 26 created
Chunk number 21 out of 26 created
Chunk number 22 out of 26 created
Chun

In [5]:
meta_fn = f"{target_path}/metadata.pkl"
save_fn = f"{target_path}/train_test_20splits.pkl"

splits = Splits(meta_fn, save_fn, n_splits=20, seed=42)
splits.create_splits()

Split number 0
Size of train subjects/indices, test subjects/indices
Number of train subjects: 1132, Number of test subjects: 60, Number of train indices: 246327, Number of test indices: 12624
Intersection size between train and test subjects: 0
Intersection size between train and test indices: 0
Split number 1
Size of train subjects/indices, test subjects/indices
Number of train subjects: 1132, Number of test subjects: 60, Number of train indices: 243924, Number of test indices: 15027
Intersection size between train and test subjects: 0
Intersection size between train and test indices: 0
Split number 2
Size of train subjects/indices, test subjects/indices
Number of train subjects: 1132, Number of test subjects: 60, Number of train indices: 245168, Number of test indices: 13783
Intersection size between train and test subjects: 0
Intersection size between train and test indices: 0
Split number 3
Size of train subjects/indices, test subjects/indices
Number of train subjects: 1132, Numbe